[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 困难：带 KV 缓存的注意力

实现**带 KV 缓存的多头注意力**，以高效进行自回归生成。

在大语言模型推理时，每一步重新计算全部键和值的投影会浪费计算资源。
**KV 缓存**保存已经计算过的 K 和 V 张量，因此只需对新增词元进行投影。

### 函数或类接口
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — 新增词元
        # cache: None 或 (K_past, V_past)，两者形状均为 (B, num_heads, S_past, d_k)
        # 返回: (output, (K_all, V_all))
```

### 实现要求
- 必须继承 `nn.Module`。
- `self.W_q`、`self.W_k`、`self.W_v` 和 `self.W_o`：使用 `nn.Linear` 实现投影。
- 当 `cache=None`（预填充阶段）时：应用**因果掩码**，并将全部 K/V 作为缓存返回。
- 当提供 `cache`（解码阶段）时：将新 K/V 与缓存拼接；单词元解码时无需因果掩码。
- 增量解码的结果必须与完整前向计算**一致**。

### 核心思路
```
预填充：[t0 t1 t2 t3] → 完整因果注意力 → cache = (K_{0:3}, V_{0:3})
解码：  [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
解码：  [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

In [ ]:
# 在 Colab 中安装 torch-judge（JupyterLab/Docker 中会跳过）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ 在此编写你的实现

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # 初始化 W_q、W_k、W_v 和 W_o

    def forward(self, x, cache=None):
        # 1. 从 x 投影得到 Q、K 和 V
        # 2. 重塑为多头形式：(B, num_heads, S, d_k)
        # 3. 若存在缓存，将新 K/V 与缓存中的 K/V 拼接
        # 4. 计算注意力（预填充阶段需要因果掩码）
        # 5. 返回 (output, (K_all, V_all))
        pass

In [ ]:
# 🧪 调试
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# 完整前向计算
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# 增量计算：预填充 4 个词元，再逐次解码 1 个词元
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

In [ ]:
# ✅ 提交
from torch_judge import check
check('kv_cache')